# Compositional Dynamic Depth — v6 CompCat Engine

**Testing whether dynamic depth in gated autoencoders reflects the compositional structure of the data-generating process.**

Based on G. Ruffini's Lie-pseudogroup compositional symmetry theory ([arXiv:2510.10586](https://arxiv.org/abs/2510.10586)).

## Key Idea

The jointed-cat renderer has a **5-level compositional hierarchy**:

| Level | Group | Description |
|-------|-------|-------------|
| 1 | SO(1)¹⁶ | Pose — 16 joint angles + head roll |
| 2 | R⁶ | Appearance — colour, thickness, eyes, stripes |
| 3 | R²×SO(3) | Placement — position + full 3D rotation |
| 4 | SE(2)×R⁺ | Camera — observer zoom/pan/rotation |
| 5 | R¹ | Background — uniform greyscale |

**Prediction:** A gated autoencoder trained on data with *d* active compositional levels should need ~*d* open gates for optimal reconstruction.

## v6 Renderer Features
- 2× supersampling anti-aliasing
- 3D sphere eye projection with per-eye visibility
- Ball-on-rod head model with SO(3) body rotation
- Drop shadows, body outlines, depth-ordered limbs
- Smooth Bézier tail, paw toe details

## v6 Gated ResNet
- Binary multiplicative gates (Gumbel-Softmax)
- Learned bypass constants (gate OFF → constant output, truly destroys info)
- AvgPool + StdPool gate conditioning

## 0. Setup

In [ ]:
import subprocess, sys, os

# Clone the repo
if not os.path.isdir('Compositional-NNs'):
    subprocess.run(['git', 'clone', 'https://github.com/giulioruffini/Compositional-NNs.git'], check=True)
os.chdir('Compositional-NNs')

# Install only what's needed (don't reinstall torch on Kaggle!)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm', 'scipy'], check=True)

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

## 1. Preview the v6 Renderer

In [ ]:
from compositional_cat_v2 import JointedCat, make_sample_grid, CONDITIONS, LEVEL_PARAMS
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Show the full hierarchy grid
grid = make_sample_grid(img_size=192, n_per_condition=8, seed=42)

fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(np.array(grid))
ax.set_title('Compositional Hierarchy — v6 CompCat Engine', fontsize=14)

# Label rows
cond_names = list(CONDITIONS.keys())
for i, name in enumerate(cond_names):
    n_lev = len(CONDITIONS[name])
    ax.text(-10, i * 192 + 96, f'{name}\n(L={n_lev})', fontsize=8,
            ha='right', va='center', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('v6_hierarchy_grid.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTotal parameters: {sum(len(v) for v in LEVEL_PARAMS.values())}')
for level, params in sorted(LEVEL_PARAMS.items()):
    print(f'  Level {level}: {len(params)} params — {", ".join(params.keys())}')

## 2. Detail View: Renderer Features

In [ ]:
cat = JointedCat()
size = 384

cases = [
    ('Default pose',    {}),
    ('Walking',         {'spine_1': 0.3, 'leg_fl_upper': -0.5, 'leg_fr_upper': 0.3,
                         'leg_bl_upper': 0.3, 'leg_br_upper': -0.5,
                         'leg_fl_lower': -0.4, 'leg_br_lower': -0.4,
                         'head_pan': 0.2, 'head_roll': 0.15, 'tail_0': 0.4}),
    ('Stretching',      {'spine_1': -0.4, 'spine_2': -0.3,
                         'leg_fl_upper': 0.4, 'leg_fl_lower': -0.8,
                         'leg_fr_upper': 0.4, 'leg_fr_lower': -0.8,
                         'head_tilt': 0.4, 'head_roll': -0.2,
                         'tail_0': 0.8, 'tail_1': 0.6}),
    ('Striped + rotated', {'root_elevation': 0.5, 'root_roll': 0.3, 'head_pan': -0.3,
                           'body_hue': 0.05, 'body_sat': 0.8, 'stripe_intensity': 0.6}),
    ('Full rotation',   {'root_angle': 2.5, 'root_elevation': 0.3,
                         'spine_1': 0.2, 'head_pan': -0.4, 'head_roll': 0.3,
                         'body_hue': 0.12, 'body_sat': 0.5}),
]

fig, axes = plt.subplots(1, len(cases), figsize=(4*len(cases), 4))
for i, (name, overrides) in enumerate(cases):
    cat.set_defaults()
    for k, v in overrides.items():
        cat.params[k] = v
    img = cat.render(img_size=size)
    axes[i].imshow(np.array(img))
    axes[i].set_title(name, fontsize=10)
    axes[i].axis('off')
plt.suptitle('v6 Renderer Detail', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('v6_detail.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Architecture Inspection

In [ ]:
from gated_resnet import GatedAutoencoder

# Match training config
IMG_SIZE = 128
LATENT_DIM = 4
BASE_CH = 32
N_STAGES = 4
N_BLOCKS = 2

model = GatedAutoencoder(
    img_size=IMG_SIZE,
    latent_dim=LATENT_DIM,
    base_channels=BASE_CH,
    n_blocks_per_stage=N_BLOCKS,
    n_stages=N_STAGES,
    gated=True,
    gate_init_bias=0.0,
    gate_tau=1.0,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_gated = model.n_gated_layers
print(f'Architecture: {N_STAGES} stages × {N_BLOCKS} blocks = {n_gated} gated layers')
print(f'Total parameters: {n_params:,}')
print(f'Latent: {LATENT_DIM} channels (spatial)')
print(f'Channel progression: {BASE_CH}', end='')
for s in range(1, N_STAGES):
    print(f' → {BASE_CH * 2**s}', end='')
print()

# Quick forward pass test
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
x_recon, z, gates = model(x)
print(f'\nForward pass: input {tuple(x.shape)} → latent {tuple(z.shape)} → output {tuple(x_recon.shape)}')
print(f'Gate values: {[f"{g.mean().item():.2f}" for g in gates]}')

## 4. Per-Condition Training

Train **one model per condition**. Each independently discovers how many gates it needs.

### Hyperparameters

In [ ]:
# ─── Training config ───
N_TRAIN = 3000        # samples per condition
N_EVAL = 500          # evaluation samples
N_EPOCHS = 80         # training epochs
BATCH_SIZE = 64
LR = 1e-3
GATE_PENALTY = 0.0005 # gate L1 penalty (same for all conditions)
GATE_WARMUP = 20      # epochs to ramp penalty from 0 → max

# Penalty sweep (optional — uncomment to try multiple values)
# PENALTIES = [0.0003, 0.0005, 0.001, 0.002]
PENALTIES = [GATE_PENALTY]

print(f'Training config:')
print(f'  {N_TRAIN} samples/condition × 6 conditions')
print(f'  {N_EPOCHS} epochs, batch size {BATCH_SIZE}')
print(f'  Gate penalty: {PENALTIES}')
print(f'  Warmup: {GATE_WARMUP} epochs')
print(f'  Device: {DEVICE}')

In [ ]:
import time
from train_and_evaluate import (
    CatDataset, train, evaluate_gates,
    plot_results, plot_training_histories
)
from torch.utils.data import DataLoader

eval_conditions = ['Static', 'PoseOnly', 'PoseAppearance',
                   'PosAppPlace', 'PosAppPlaceCam', 'Everything']

all_sweep_results = {}  # penalty → results

for penalty_val in PENALTIES:
    print(f'\n{"="*60}')
    print(f'PENALTY λ = {penalty_val}')
    print(f'{"="*60}')

    out_dir = f'results_lambda_{penalty_val:.4f}'
    os.makedirs(out_dir, exist_ok=True)

    all_results = []
    all_histories = {}
    t_start = time.time()

    for cond in eval_conditions:
        n_lev = len(CONDITIONS[cond])
        print(f'\n{"─"*50}')
        print(f'Training: {cond} (levels={CONDITIONS[cond]}, dim={n_lev})')
        print(f'{"─"*50}')

        # Fresh model
        model = GatedAutoencoder(
            img_size=IMG_SIZE, latent_dim=LATENT_DIM,
            base_channels=BASE_CH, n_blocks_per_stage=N_BLOCKS,
            n_stages=N_STAGES, gated=True,
            gate_init_bias=0.0, gate_tau=1.0,
        )

        # Dataset
        train_dataset = CatDataset(cond, n_samples=N_TRAIN,
                                    img_size=IMG_SIZE, seed=42)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                   shuffle=True, num_workers=2)

        # Train
        t0 = time.time()
        history = train(model, train_loader,
                        n_epochs=N_EPOCHS, lr=LR,
                        gate_penalty_max=penalty_val,
                        gate_warmup_epochs=GATE_WARMUP,
                        device=DEVICE, verbose=True)
        dt = time.time() - t0
        all_histories[cond] = history

        # Save model
        torch.save(model.state_dict(),
                   os.path.join(out_dir, f'model_{cond}.pt'))

        # Evaluate
        r = evaluate_gates(model, cond, n_samples=N_EVAL,
                           img_size=IMG_SIZE, device=DEVICE)
        all_results.append(r)

        print(f'  → D_eff = {r["effective_depth_mean"]:.2f} ± '
              f'{r["effective_depth_std"]:.2f} | '
              f'Recon = {r["recon_error_mean"]:.4f} | '
              f'Time: {dt:.0f}s')
        gate_str = ' '.join(f'{g:.2f}' for g in r['gate_means'])
        print(f'    Gates: [{gate_str}]')

    total_time = time.time() - t_start
    print(f'\nTotal training time: {total_time/60:.1f} min')
    all_sweep_results[penalty_val] = (all_results, all_histories)

## 5. Results

In [ ]:
import json

for penalty_val, (results, histories) in all_sweep_results.items():
    out_dir = f'results_lambda_{penalty_val:.4f}'
    os.makedirs(out_dir, exist_ok=True)

    # Save JSON
    with open(os.path.join(out_dir, 'gate_analysis.json'), 'w') as f:
        json.dump(results, f, indent=2)

    # Main results plot
    plot_results(results, output_dir=out_dir)
    plot_training_histories(histories, output_dir=out_dir)

    # Display
    fig = plt.figure(figsize=(14, 10))
    img = plt.imread(os.path.join(out_dir, 'dynamic_depth_results.png'))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Dynamic Depth Results (λ={penalty_val})', fontsize=14)
    plt.show()

    fig = plt.figure(figsize=(14, 5))
    img = plt.imread(os.path.join(out_dir, 'training_curves.png'))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Training Curves (λ={penalty_val})', fontsize=14)
    plt.show()

## 6. Summary Table

In [ ]:
for penalty_val, (results, _) in all_sweep_results.items():
    print(f'\n{"="*60}')
    print(f'SUMMARY: λ = {penalty_val}')
    print(f'{"="*60}')
    print(f'{"Condition":<20s} {"Levels":>7s} {"D_eff":>8s} {"±":>6s} {"Recon":>8s}')
    print('-' * 52)

    d_effs = []
    for r in results:
        print(f'{r["condition"]:<20s} {r["n_active_levels"]:>7d} '
              f'{r["effective_depth_mean"]:>8.2f} '
              f'{r["effective_depth_std"]:>6.2f} '
              f'{r["recon_error_mean"]:>8.4f}')
        d_effs.append(r['effective_depth_mean'])

    # Statistics
    n_levs = np.array([r['n_active_levels'] for r in results])
    d_arr = np.array(d_effs)
    monotone = all(d_effs[i] <= d_effs[i+1] + 0.5 for i in range(len(d_effs)-1))

    print(f'\nMonotone D_eff: {"YES ✓" if monotone else "NOT CLEARLY"}')
    print(f'D_eff spread (Everything − Static): {d_effs[-1] - d_effs[0]:.2f}')

    if n_levs.std() > 0 and d_arr.std() > 0:
        corr = np.corrcoef(n_levs, d_arr)[0, 1]
        print(f'Pearson r(n_levels, D_eff): {corr:.3f}')

## 7. Reconstruction Examples

In [ ]:
# Show original vs reconstructed for each condition
from compositional_cat_v2 import sample_params

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Original (top) vs Reconstruction (bottom)', fontsize=14, fontweight='bold')

rng = np.random.RandomState(123)
penalty_val = PENALTIES[0]
results, _ = all_sweep_results[penalty_val]

for col, cond in enumerate(eval_conditions):
    # Generate a test image
    cat = JointedCat()
    cat.params = sample_params(CONDITIONS[cond], rng)
    img_pil = cat.render(img_size=IMG_SIZE)
    img_np = np.array(img_pil, dtype=np.float32) / 255.0
    x = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    # Load model (retrain or use last state — here we just use fresh eval)
    model = GatedAutoencoder(
        img_size=IMG_SIZE, latent_dim=LATENT_DIM,
        base_channels=BASE_CH, n_blocks_per_stage=N_BLOCKS,
        n_stages=N_STAGES, gated=True,
    ).to(DEVICE)
    out_dir = f'results_lambda_{penalty_val:.4f}'
    model_path = os.path.join(out_dir, f'model_{cond}.pt')
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    with torch.no_grad():
        x_recon, _, gates = model(x)

    # Display
    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f'{cond}\n(L={len(CONDITIONS[cond])})', fontsize=8)
    axes[0, col].axis('off')

    recon_np = x_recon[0].cpu().permute(1, 2, 0).numpy().clip(0, 1)
    axes[1, col].imshow(recon_np)
    d_eff = sum(g.mean().item() for g in gates)
    axes[1, col].set_title(f'D_eff={d_eff:.1f}', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('reconstruction_examples.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Gate Heatmap (Detailed)

In [ ]:
penalty_val = PENALTIES[0]
results, _ = all_sweep_results[penalty_val]

gate_matrix = np.array([r['gate_means'] for r in results])
conditions = [r['condition'] for r in results]
n_levels = [r['n_active_levels'] for r in results]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
im = ax.imshow(gate_matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

# Annotate each cell
for i in range(gate_matrix.shape[0]):
    for j in range(gate_matrix.shape[1]):
        val = gate_matrix[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color=color)

ax.set_xlabel('Gated Layer Index', fontsize=12)
ax.set_ylabel('Condition', fontsize=12)
ax.set_yticks(range(len(conditions)))
ax.set_yticklabels([f'{c} (L={n})' for c, n in zip(conditions, n_levels)], fontsize=9)
ax.set_xticks(range(gate_matrix.shape[1]))
ax.set_title(f'Gate Activation Heatmap (λ={penalty_val})', fontsize=14)
plt.colorbar(im, ax=ax, label='Mean gate probability')
plt.tight_layout()
plt.savefig('gate_heatmap_detailed.png', dpi=150, bbox_inches='tight')
plt.show()

print('Expected pattern: "staircase" — more levels → more active gates')

## 9. Interpretation

### What to look for:

1. **Gate heatmap should show a staircase**: Static uses few gates, Everything uses most.
2. **D_eff should increase monotonically** with the number of active generative levels.
3. **Static should have D_eff ≈ 0**: with one fixed image, the model can memorize everything via bypass constants.
4. **Geometric levels (1-4) should drive depth more than appearance/background (2, 5)**: spatial variation requires spatial processing (conv layers), while colour changes can be handled by 1×1 transforms.

### Theory connection:
The Lie-pseudogroup flag `H₀ ⊃ H₁ ⊃ ... ⊃ H₅` predicts that each level of compositional complexity adds one "degree of depth" to the optimal representation. The gated autoencoder should discover this structure by learning to open gates selectively based on the input's compositional complexity.